# Create NHANES dataframe for Microsim

Goal: This notebook creates a dataframe with all NHANES person information for use by Microsim

JB: (don't forget to hop into the conda3 virtual environment — source activate base)

To Dos:
<ul>
    <li> make sure to add smoking to the risk factor models</li>
    <li> creatinine/GFR</li>
    <li> build out the nhanes file loader method below into something prettier</li>
</ul>

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import shutil

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

import wget
import os.path
import shutil

#!pip install wget #CD added, if needed...

baseDir = "/Users/deligkaris.1/OneDrive - The Ohio State University Wexner Medical Center/MICROSIM/NOTEBOOKS/DATA/"
os.chdir(baseDir)

### Function to download NHANES files for specific variables

In [2]:
#https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/BPX_F.xpt

def downloadYear(year, stem):
    plusOne = year + 1
    dirName = "NHANES-RAW-DATA/" + str(year) + "-" + str(plusOne)
    os.chdir(dirName)
    #url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/" + str(year) + "-" + str(plusOne) + "/" + stem + ".XPT" #
    url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/" + str(year) + "/DataFiles/" + stem + ".XPT" #CD added, the link has changed...
    #print(url)#CD I do not need to see the files....if there is an error it will show up....

    fileName = wget.download(url, stem + ".XPT")  
    # Overwrite file if already exists
    if os.path.exists(fileName):
        shutil.move(fileName,stem + ".XPT")
    os.chdir(baseDir)
    return pd.read_sas(os.path.join(dirName, stem + '.XPT'))
   
def loadBaseFilesForYear(year, suffix):
    # looks like OGTT dropped in 2018...
    baseFileStems = ['DEMO','MCQ', 'SMQ', 'DIQ','GHB', 'HDL', 'TCHOL', 'BMX', 'BPX', 'TRIGLY', 'BPQ', 'BIOPRO']
    for baseStem in baseFileStems:
        if suffix == '':
            stem = baseStem
        else:
            stem = baseStem+'_'+suffix
        if not ((baseStem in ["GHB","HDL","TCHOL"]) & (year<2005)):
            if year==2021:
                if baseStem=="BPX": 
                    baseStem="BPXO" #CD: things changed in 2021...
                    stem = baseStem+'_'+suffix
                #print(stem)
            downloadYear(year, stem)
    
    
# loaded for prior years manually
#loadBaseFilesForYear(1999, '') #CD: I think the years 1999, 2001, 2003 were commented out here because some 
#file stems, GHB, HDL, TCHOL and perhaps others too do not exist for those years
#loadBaseFilesForYear(2001, 'B')
#loadBaseFilesForYear(2003, 'C')
loadBaseFilesForYear(2005, 'D')
loadBaseFilesForYear(2007, 'E')
loadBaseFilesForYear(2009, 'F')
loadBaseFilesForYear(2011, 'G')
loadBaseFilesForYear(2013, 'H')
loadBaseFilesForYear(2015, 'I')
loadBaseFilesForYear(2017, 'J')
#loadBaseFilesForYear(2021, 'L') #the RXQ_RX_L file from 2021 is significantly different than the previous ones...
#https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/DEMO_D.xpt

In [3]:
def loadFilesForStems(stems):
    stemIndex = 0  
    for year in list(range(1999,2018,2)): # + [2021]:
        downloadYear(year, stems[stemIndex])
        stemIndex += 1
 
#CD: tried including the subsequent year-files but did not work

glucoseFileStems = ['LAB10AM', 'L10_2_B','L10AM_C', 'GLU_D', 'GLU_E', 'GLU_F', 'GLU_G', 'GLU_H', 'GLU_I', 'GLU_J']#, 'GLU_L']
loadFilesForStems(glucoseFileStems)
    
paqFileStems = ['PAQ', 'PAQ_B','PAQ_C', 'PAQ_D', 'PAQ_E', 'PAQ_F', 'PAQ_G', 'PAQ_H', 'PAQ_I', 'PAQ_J']#, 'PAQ_L' ]
loadFilesForStems(paqFileStems)

rxFileStems = ['RXQ_RX', 'RXQ_RX_B','RXQ_RX_C', 'RXQ_RX_D', 'RXQ_RX_E', 'RXQ_RX_F', 'RXQ_RX_G', 'RXQ_RX_H', 'RXQ_RX_I', 
               'RXQ_RX_J']#, 'RXQ_RX_L' ]
loadFilesForStems(rxFileStems)

alcFileStems = ['ALQ', 'ALQ_B','ALQ_C', 'ALQ_D', 'ALQ_E', 'ALQ_F', 'ALQ_G', 'ALQ_H', 'ALQ_I', 'ALQ_J']#, 'ALQ_L' ]
loadFilesForStems(alcFileStems)
    
bpqFileStems = ['BPQ', 'BPQ_B','BPQ_C', 'BPQ_D', 'BPQ_E', 'BPQ_F', 'BPQ_G', 'BPQ_H', 'BPQ_I', 'BPQ_J']#, 'BPQ_L' ]
loadFilesForStems(bpqFileStems)

metabolicPanelStems = ['LAB18', 'L40_B', 'L40_C', 'BIOPRO_D', 'BIOPRO_E', 'BIOPRO_F', 'BIOPRO_G', 'BIOPRO_H', 
                       'BIOPRO_I', 'BIOPRO_J'] #, 'BIOPRO_L']
loadFilesForStems(metabolicPanelStems)

albCreatinineStemps = ['LAB16', 'L16_B', 'L16_C', 'ALB_CR_D', 'ALB_CR_E', 'ALB_CR_F', 'ALB_CR_G', 'ALB_CR_H', 'ALB_CR_I', 
                       'ALB_CR_J'] #, 'ALB_CR_L']
loadFilesForStems(albCreatinineStemps)

In [4]:
# level one gets us to cardiovascular meds...these are the level 2 value-counts
#drugDatafile = downloadYear(1999, 'RXQ_DRUG') #CD: no longer exists
#search for rxq_drug here: https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx
drugDatafile = downloadYear(1988, 'RXQ_DRUG') #CD added, there is no other rxq_drug file
drugDatafile['antiHypertensive'] = 0
drugDatafile['statin'] = 0
drugDatafile['otherLipidLowering'] = 0
drugDatafile.loc[((drugDatafile.RXDDCI1A==40) & (drugDatafile.RXDDCI1B.isin([42, 43, 44,47,48, 49,53,55,56]))), 'antiHypertensive'] = 1
drugDatafile.loc[((drugDatafile.RXDDCI1A==358) & (drugDatafile.RXDDCI1B == 19) & (drugDatafile.RXDDCI1C==173)), 'statin'] = 1
drugDatafile.loc[((drugDatafile.RXDDCI1A==358) & (drugDatafile.RXDDCI1B == 19) & (drugDatafile.RXDDCI1C.isin([174,241,252,316,317]))), 'otherLipidLowering'] = 1

In [5]:
#CD: I think these are just for double checking...

#demo17 = pd.read_sas("/Users/burke/Documents/research/bpCog/nhanes/sourceData/nhanesRawData/2017-2018/DEMO_J.XPT")
#meds17 = pd.read_csv("/Users/burke/Documents/research/bpCog/nhanes/sourceData/nhanesRawData/2017-2018/medSummary.csv")

#rx17 = pd.read_sas("/Users/burke/Documents/research/bpCog/nhanes/sourceData/nhanesRawData/2013-2014/RXQ_RX_H.XPT")

#lab01 = pd.read_sas("/Users/burke/Documents/research/bpCog/nhanes/sourceData/nhanesRawData/2001-2002/L40_B.XPT")

#lab01.head()

### Load All NHANES Data Files and append into a single dataset



In [6]:
''' iterate through the root directory which has subdiretxories (with year names) for each annual collection of 
raw NHANES data files '''

root = next(os.walk("NHANES-RAW-DATA"))[0]
yearDirs = next(os.walk("NHANES-RAW-DATA"))[1]
yearDirs.remove('2003-2004') #CD: I am excluding these directories here because they do not have complete information
yearDirs.remove('2001-2002')
yearDirs.remove('1988-1989')
yearDirs.remove('1999-2000')
yearDirsWithRoot = [root + os.sep + year for year in yearDirs]


# loads the files from a year — starts with the demographics files and then merges in all of the other subfiles in the directory

def loadYear(dir):
    demographics = getDemographics(dir)
    otherFiles = next(os.walk(dir))[2]
    otherFiles.remove(demographics)
    
    demoSAS = pd.read_sas(dir + os.sep + demographics)
    for other in otherFiles:
        if other[-3:] == "csv":
            otherSAS = pd.read_csv(dir + os.sep + other)
            demoSAS = demoSAS.merge(otherSAS, how='left', on=['SEQN'])
        elif other.startswith("RXQ"):
            continue
        elif other[-3:].lower() == "xpt":
            #print(other) #CD no need...
            otherSAS = pd.read_sas(dir + os.sep + other)
            demoSAS = demoSAS.merge(otherSAS, how='left', on=['SEQN'])
    #year = dir[dir.find(os.sep)+1:dir.find("-")]
    folder_name = dir.split('/')[-1]  # result: "2011-2012"
    year = folder_name.split('-')[0]  # result: "2011"
    #print(year) #CD no need to see it...
    demoSAS['year'] = year
    
    # left off here — need to get date of death into this by pulling it from the file year in teh loop
    columnNameMappings = {"RIDAGEYR" : "age", "RIAGENDR" :  "gender",  "RIDRETH1" : "raceEthnicity",
                          "BPXSY1" : "sbp1", "BPXSY2" : "sbp2", "BPXSY3" : "sbp3", "BPXSY4" : "sbp4", "BPXDI1" : "dbp1",
                        "BPXDI2": "dbp2", "BPXDI3" : "dbp3", "BPXDI4" : "dbp4", "LBXGH" : "a1c", "LBDHDD" : "hdl",
                        "LBXTC" : "tot_chol", "SMQ040" : "currentSmoker", "SMQ020" : "quantitySmoked","BMXWT" : "weight",
                          "BMXBMI" : "bmi", "BMXWAIST" : "waist", "LBDLDL" : "ldl", "LBXTR" : "trig",  
                          "DMDYRSUS" : "timeInUS", "MORTSTAT" : "diedBy2015", "UCOD_LEA" : "ucod", 
                         "MCQ160E" :"selfReportMI", "MCQ180E" : "selfReportMIAge", 
                            "MCQ160F" : "selfReportStroke", "MCQ180F" : "selfReportStrokeAge", 
                          "PERMTH_I" : "monthsToDeath", "PERMTH_E" : "monthsToDeath2", "DMDEDUC2" : 'education',
                         'antiHypertensive' : 'antiHypertensive', 'statin' : 'statin', 'otherLipidLowering' : 'otherLipidLowering',
                         'PAQ605' : 'vigorousWork', 'PAQ620' : 'moderateWork', 'PAQ635' : 'walkBicyle', 
                         'PAQ650' : 'vigorousRecreation', 'PAQ665' : 'moderateRecreation',
                         'PAD020' : 'walkBikeOld', 'PAD200' : 'vigorousOld', 'PAD320' : 'moderateOld',
                         'ALQ130' : 'alcoholDrinksPerDay', 'ALQ100' : 'alcoholLastYear',
                         'ALQ110' : 'alcoholLifetime', 'RIDSTATR' : 'interviewStatus', 
                         'BPQ020' : 'selfReportHtn', 'BPQ050A': 'selfReportCurrentHtnMed',
                         'URXUMA' : 'urineAlbumin', 'LBXSCR' : 'serumCreatinine'}
    
    alternateNames = {'LBXTC' : ['LBXTC_x' ], 'LBDHDD' : ['LBDHDL','LBXHDD', 'LBDHDL_x'], 'LBXSCR' : ['LBDSCR']}
    
    demoSAS.to_csv(str(year + "Before.csv"))
    # we have a couple variable names that get remapped to differnet names...this will remap them all to consistent names
    for key in alternateNames.keys():
        if key not in demoSAS:
            print(str(key) + " is missing from year: " + str(year))
            for value in alternateNames[key]:
                #print ("checking: " + value) #CD: no need to see it..
                if value in demoSAS:
                    #print ("rename: " + value + " to key: " + key) #CD: no need to see it...
                    demoSAS.rename(columns={value: key}, inplace=True)

    demoSAS.to_csv(str(year + "InBetween.csv"))

    
    columnsToKeep = list(columnNameMappings.keys())
    columnsToKeep.extend(["SEQN", "WTINT2YR", "WTMEC2YR", "SDMVPSU", "SDMVSTRA", "year"])
    
    # some of the factors that we car eabout are missing in given years...add them and make them null here.
    for value in columnsToKeep:
        if value not in demoSAS:
            demoSAS[value] = np.nan
    
    demoSAS = demoSAS[columnsToKeep]
    demoSAS.rename(columns=columnNameMappings, inplace=True)
    demoSAS.to_csv(str(year) + ".csv")
    
    return demoSAS

def getDemographics(dir):
    files = next(os.walk(dir))[2]
    for file in files:
        if file.startswith("DEMO"):
            return file
    raise Exception("Unable to find demographics file in directory: " + dir)
        
    
# the med files are going to be our exception — for all of the other files, we just need to get a single row per person
# for meds, there are multiple rows per person...so, we'll build a summary file that extracts the med fieds that we care 
# about into a summary file with one line per person
def convertMedsToMedSummaryFiles(dir):
    allFiles = next(os.walk(dir))[2]
    for file in allFiles:
        if file.startswith('RXQ_RX'):
            #print(f"{dir} + {file}")

            medsFile = pd.read_sas(os.path.join(dir, file))
            summary = medsFile.merge(drugDatafile, on='RXDDRGID').groupby('SEQN')[['SEQN', 'antiHypertensive', 'statin', 'otherLipidLowering']].sum()
            summary.to_csv(os.path.join(dir, "medSummary.csv"))
    
[convertMedsToMedSummaryFiles(x) for x in yearDirsWithRoot]
yearDataFrames = [loadYear(x) for x in yearDirsWithRoot]
combined = pd.concat(yearDataFrames, sort=True)

combined['anyPhysicalActivity'] = 0
combined.loc[(combined.vigorousWork==1) | (combined.moderateWork==1) | (combined.walkBicyle==1) | (combined.vigorousRecreation==1) | (combined.moderateRecreation==1),'anyPhysicalActivity'] = 1
combined.loc[(combined.walkBikeOld==1) | (combined.vigorousOld==1) | (combined.moderateOld==1),'anyPhysicalActivity'] = 1
combined.drop(labels=['vigorousWork', 'moderateWork', 'walkBicyle', 'vigorousRecreation', 'moderateRecreation', 'walkBikeOld', 'vigorousOld', 'moderateOld'], axis=1, inplace=True)
combined.education.replace(to_replace=[7,9], value=np.nan, inplace=True)


# combine the two smoking status variables into a variable that fits into ASCVD
def generateSmokingStatus(x):
    if np.isclose(x.currentSmoker, 1, 0.0001) or np.isclose(x.currentSmoker, 2, 0.0001):
        return "2"
    elif np.isclose(x.quantitySmoked, 1, 0.0001):
        return "1"
    else:
        return "0"
    
smokingStatusValues = [generateSmokingStatus(x) for _,x in combined.iterrows()]
combined = combined.assign(smokingStatus=pd.Series(data=smokingStatusValues, index=combined.index, dtype="category"))
combined.drop(labels=["currentSmoker", "quantitySmoked"],axis=1, inplace=True)
combined.raceEthnicity = combined.raceEthnicity.astype('int')
combined.raceEthnicity = combined.raceEthnicity.astype("category")
combined.education.fillna(4, inplace=True) # small amount of mnissing data...will crudely impute with temost common value
combined[['education']] = combined[['education']].astype('int')
combined.education = combined.education.astype('category')

combined['meanSBP'] = combined[['sbp1', 'sbp2', 'sbp3', 'sbp4']].mean(axis=1)
combined['meanDBP'] = combined[['dbp1', 'dbp2', 'dbp3', 'dbp4']].mean(axis=1)

combined['alcoholPerWeek'] = combined.alcoholDrinksPerDay * 7
combined.alcoholPerWeek= np.where(combined.alcoholLastYear ==2, 0, combined.alcoholPerWeek)
combined.alcoholPerWeek = np.where(combined.alcoholLifetime ==2, 0, combined.alcoholPerWeek)
combined.alcoholPerWeek = np.where(combined.alcoholDrinksPerDay > 40, np.nan, combined.alcoholPerWeek)

combined['completedInterview'] = 0
combined.loc[combined.interviewStatus==2,'completedInterview'] = 1

# there are a bunch of very low DBPs that don't correlate with SBPs at all...we're going to call those missing
# and then impute them
combined.loc[combined.meanDBP < 40, 'meanDBP'] = np.nan
combined['year'] = pd.to_numeric(combined['year'], errors='coerce') #CD added
combined['year'] = combined['year'].astype('Int64')
combined['missingSBP'] = combined.meanSBP.isna()

# get some information for anti-hypertensifes to help imputatino...
combined.loc[combined['selfReportHtn'] !=1, 'selfReportHtn'] = 0
combined.loc[combined['selfReportCurrentHtnMed'] !=1, 'selfReportCurrentHtnMed'] = 0


### Clean up some variables with muitple field names and build a smoking summary variable
#over18 = combined.loc[combined.age >= 18] #CD: commented out because we now need all NHANES people
over18 = combined #CD: this is misleading since it includes everyone but keep as is for now...


In [8]:
#CD: the less than 20 years old in the dataframe will have all of them the same category for education, smoking, alcohol and 
#physical activity.
#This is done this way so that when others create population groups by categorical variables
#age group is one of those categorical variables, it goes 0-4, 5-9, 10-14, 15-19
#will be consistent with us here and that is extremely important when we sample from NHANES to create the population
#that fits the given categories (unless of course we do not use alcohol, edication, smoking, activity to do the sampling)
#but we do later in microsim have the people less than 20 years old obtain, during the simulation, a proper 
#category for alcohol, education, alcohol and activity

microsimDir = "/Users/deligkaris.1/OneDrive - The Ohio State University Wexner Medical Center/MICROSIM/CODE/microsim"
os.chdir(microsimDir)
from microsim.risk_factors.alcohol_category import AlcoholCategory
from microsim.risk_factors.education import Education
from microsim.risk_factors.smoking_status import SmokingStatus

def assign_kids_education(row):
    if row["age"]>19:
        return row["education"]
    #elif row["age"]>13:
    #    return Education.SOMEHIGHSCHOOL.value
    else:
        return Education.LESSTHANHIGHSCHOOL.value
    
over18['education'] = over18.apply( assign_kids_education, axis=1)
 
def assign_kids_smoking(row):
    if row["age"]>19:
        return row["smokingStatus"]
    else:
        return '0' #SmokingStatus.NEVER.value

over18['smokingStatus'] = over18.apply( assign_kids_smoking, axis=1)
    
def assign_kids_alcohol(row):
    if row["age"]>19:
        return row["alcoholPerWeek"]
    else:
        return AlcoholCategory.NONE.value

over18['alcoholPerWeek'] = over18.apply( assign_kids_alcohol, axis=1)
    
def assign_kids_activity(row):
    if row["age"]>19:
        return row["anyPhysicalActivity"]
    else:
        return 1

over18['anyPhysicalActivity'] = over18.apply( assign_kids_activity, axis=1)

os.chdir(baseDir)

In [62]:
#over18.head(30)
#over18.info()

In [10]:
# from18to28yo = combined.loc[ (combined["age"]>17) & (combined["age"]<29) ].copy() 

# proportionsEdu = from18to28yo.groupby(["gender", "raceEthnicity"])["education"].value_counts(normalize=True)
# #proportionsEdu = proportionsEdu.reset_index(name='proportion')

# result_dict = proportionsEdu.unstack().to_dict(orient='index')

# def sample_child_education_category(row, lookup_dict):
#     # 1. Create the key for the dictionary lookup
#     key = (row['gender'], row['raceEthnicity'])
    
#     # 2. Get the proportions for this specific group
#     # Using .get() prevents errors if a group combination is missing
#     group_proportions = lookup_dict.get(key)
    
#     if not group_proportions:
#         return None
    
#     # 3. Use the random number to pick a category
#     rand_val = row['random_num']
#     cumulative_prob = 0
    
#     for category, proportion in group_proportions.items():
#         cumulative_prob += proportion
#         if rand_val <= cumulative_prob:
#             return category
            
#     # Fallback for floating point rounding issues (e.g., if total is 0.9999)
#     return list(group_proportions.keys())[-1]

# def fix_education(row, result_dict):
#     if row["age"]>17:
#         return row["education"]
#     else:
#         return sample_child_education_category(row, result_dict)
    
# combined['random_num'] = np.random.rand(len(combined))

# combined['education'] = combined.apply(
#     fix_education, 
#     axis=1, 
#     args=(result_dict,)
# )

In [11]:
#combined.loc[combined["age"]<5].head(45)

In [12]:
#result_dict

In [13]:
#proportionsEdu

In [14]:
#pd.crosstab(combined.selfReportHtn, combined.selfReportCurrentHtnMed) #CD: just checking..

In [15]:
#combined.selfReportHtn.value_counts() #CD just checking...

In [16]:
#combined.completedInterview.value_counts() #CD just checking...

In [17]:
# this aligns with the total nmumber that get interview or exam...https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/DEMO_J.htm
# and the 8704 aligns with the total number that completed the interview in the JAMA paper: https://cdn.jamanetwork.com/ama/content_public/journal/jama/938557/joi200090supp1_prod_1600712145.41674.pdf?Expires=1627414089&Signature=lVm5OOLRJnT8mMvVXLkb9MwC7~Bj02SLlQpYb-hRes5teUgXAARaw33lkRZtkmD-bW3wtlCThEQJITXs8erWnC~t-yWDNatjg3ZUprHTtBQjtY4Iikbnf0LW1ycXEvid8AlKUWAZAN7O94~bgBU178sgIk5Te~Bzl1kHPu-WyOPKiHGKqsmJ7TcN5ohE4nJewBcbnOpXMXHsK4yDWmkH9Ys-30HLZ2nBIRhPqAOa5EuO9Y2dY4KP1V2Nc-IN~bw-WDvLXocEcPtRCv71bL1OTAQN1Lxrp-QL4aFokowmWKuNWpVhI7KVvVJcmf74HcnHC2~41yGpi0PIaHYGWXFO4g__&Key-Pair-Id=APKAIE5G5CRDK6RD3PGA
#print(len(combined.loc[combined.year==2017])) #CD: this is just for checking probably..
#print(combined.loc[combined.year==2017].completedInterview.value_counts())

In [18]:
#CD: not sure what these are doing..

over18['htn3'] = 0
over18['htn4'] = 0

over18['meanSBP3'] = over18[['sbp1', 'sbp2', 'sbp3']].mean(axis=1)
over18.loc[((over18['meanSBP'] >= 140) | (over18['meanDBP'] >= 90) | (over18['antiHypertensive'] > 0)), 'htn4'] = 1
over18.loc[((over18['meanSBP3'] >= 140) | (over18['meanDBP'] >= 90) | (over18['antiHypertensive'] > 0)), 'htn3'] = 1


In [19]:
#CD these are probably checking..
#test2017All = combined.loc[combined.year==2017]
#test2017Adult = over18.loc[over18.year==2017] #CD: this is actually the same as above now...

In [20]:
#test2017Adult.groupby('interviewStatus')[['age', 'gender', 'sbp1']].mean()
# non examined patients are slightly mnore likely to be male and slighty more likely to be older...

In [21]:
#test2017Adult['missingSBPOnly'] = test2017Adult['missingSBP'] & (test2017Adult['interviewStatus'] == 2) #CD: these are just for checking..
#test2017Adult.groupby('missingSBPOnly')[['age', 'gender', 'meanDBP', 'bmi', 'ldl', 'waist', 'tot_chol', 'smokingStatus']].mean()

In [22]:
#pd.crosstab(test2017Adult.missingSBP, test2017Adult.interviewStatus) #CD: just checking probably...

In [23]:
# ok...it doesn't look like the problem is 3 measures vs. 4...that's not going to be a source of bias...
#pd.crosstab(test2017Adult.htn3, test2017Adult.htn4)

In [24]:
# 3 measures vs. 4 meaures isn't our problem...
#plt.scatter(test2017Adult.meanSBP3, test2017Adult.meanSBP)

In [25]:
#over18.groupby('year').apply(lambda x : x.count()) / len(over18)

In [26]:
#print(over18.columns[over18.columns.duplicated()])

In [27]:
regressDF = over18.copy() #CD: remember this includes everyone now
#regressDF.drop("year", axis=1, inplace=True)
regressDF = pd.get_dummies(regressDF, columns =['raceEthnicity', 'smokingStatus', 'education'])
regressDF.drop(labels=['SEQN', 'WTMEC2YR', 'SDMVPSU','SDMVSTRA','interviewStatus' ],axis=1, inplace=True)
regressDF.drop(labels=['sbp1', 'sbp2','sbp3', 'sbp4','dbp1', 'dbp2', 'dbp3', 'dbp4' ],axis=1, inplace=True)
regressDF.drop(labels=['alcoholDrinksPerDay','alcoholLastYear', 'alcoholLifetime' ],axis=1, inplace=True)

regressDF.year = regressDF.year.astype('Int64') #CD: converted some of the types to Int64..
regressDF.completedInterview = regressDF.completedInterview.astype('Int64')
regressDF.missingSBP = regressDF.missingSBP.astype('Int64')
regressDF.selfReportStroke = regressDF.selfReportStroke==1
regressDF.selfReportMI = regressDF.selfReportMI==1
regressDF.ucod.replace({b'010' : '1', b'005' : 2}, inplace=True)
#regressDF.ucod.loc[regressDF['ucod'].isin([b'002', b'001',b'003',b'007',b'004',b'008',b'006',b'009' ])]=3
regressDF.loc[regressDF['ucod'].isin([b'002', b'001',b'003',b'007',b'004',b'008',b'006',b'009' ]),'ucod']=3 #CD added
#regressDF.ucod.loc[regressDF['ucod'].isin([b'',b'.'])]=0
regressDF.loc[regressDF['ucod'].isin([b'',b'.']),'ucod']=0 #CD added
regressDF.ucod.fillna(value=0, inplace=True)
regressDF.ucod = regressDF.ucod.astype('Int64')
regressDF.selfReportMI = regressDF.selfReportMI.astype('Int64')
regressDF.selfReportStroke = regressDF.selfReportStroke.astype('Int64')
regressDF.diedBy2015 = regressDF.diedBy2015.fillna(0)
regressDF.diedBy2015 = regressDF.diedBy2015.astype('Int64')
regressDF.selfReportHtn = regressDF.selfReportHtn.astype('Int64')
regressDF.selfReportCurrentHtnMed = regressDF.selfReportCurrentHtnMed.astype('Int64')
regressDF['jnc8'] = 0
regressDF.loc[((regressDF.meanSBP > 140) | (regressDF.meanDBP > 90)), 'jnc8'] = 1
regressDF.reset_index(inplace=True)
regressDF.to_stata('nonImputedDataset.dta')

In [28]:
completeCase = regressDF[['WTINT2YR', 'a1c', 'age', 'antiHypertensive', 'bmi','gender', 'hdl', 'ldl', 'otherLipidLowering',
           'statin', 'tot_chol', 'trig', 'ucod', 'waist', 'weight', 'year',
       'anyPhysicalActivity', 'meanSBP', 'meanDBP', 'alcoholPerWeek',
       'raceEthnicity_1', 'raceEthnicity_2', 'raceEthnicity_3',
       'raceEthnicity_4', 'raceEthnicity_5', 'smokingStatus_0',
       'smokingStatus_1', 'smokingStatus_2', 'education_1', 'education_2',
       'education_3', 'education_4', 'education_5', 'serumCreatinine', 'urineAlbumin']].dropna()
completeCase = completeCase.merge(regressDF[list(set(regressDF.columns)- (set(completeCase.columns)))], how='left', left_index=True, right_index=True)
completeCase.to_stata('completeCaseDataset.dta')

### Use MICE to impute missing data



In [29]:
import statsmodels.imputation.mice as mice
import statsmodels.discrete.discrete_model as dm

def ols_formula(all_variables, dependent_var, vars_to_remove):
    copied_vars = all_variables.copy()
    copied_vars.remove(dependent_var)
    vars_for_formula = [var for var in copied_vars if var not in vars_to_remove ]
    fml = ''
    first = True
    for col in vars_for_formula:
        if first:
            fml = fml + col
        else:
            fml = fml + ' + ' + col
        first = False
    return fml

cols = ['a1c', 'age', 'antiHypertensive', 'bmi', 'gender', 'hdl',  'ldl',
       'otherLipidLowering', 'selfReportMI', 'selfReportStroke', 'statin',
       'timeInUS', 'tot_chol', 'trig', 'waist', 'weight', 'year',
       'anyPhysicalActivity', 'meanSBP', 'meanDBP', 'alcoholPerWeek',
       'raceEthnicity_1',
       'raceEthnicity_2', 'raceEthnicity_3', 'raceEthnicity_4',
       'raceEthnicity_5', 'smokingStatus_0', 'smokingStatus_1',
       'smokingStatus_2', 'education_1', 'education_2', 'education_3',
       'education_4', 'education_5', 'serumCreatinine', 'urineAlbumin']

#CD: mice complains about columns with all values missing but I do not see any column like that...
#for col in cols+ ['jnc8', 'selfReportHtn', 'selfReportCurrentHtnMed','diedBy2015']:
#    print(f"{col} = {regressDF[col].notnull().sum()}")

#CD: statsmodels do not recognize Int64 I used above...so convert those to float for now... 
int64_cols = regressDF.select_dtypes(include=['Int64', 'Int32']).columns 
regressDF[int64_cols] = regressDF[int64_cols].astype(float)

#valid_cols = [col for col in cols if regressDF[col].notnull().sum() > 0] + ['jnc8', 'selfReportHtn', 
                                                                            #'selfReportCurrentHtnMed','diedBy2015']#CD added
#CD: by trial and error I found that mice needs these columns...
colsNeeded = cols + ['jnc8', 'selfReportHtn', 'selfReportCurrentHtnMed','diedBy2015','WTINT2YR'] 

imputedData = mice.MICEData(regressDF[colsNeeded])
#imputedData = mice.MICEData(regressDF)

# have to leave LDL and trig out of each other's imputations...since their missingness is highly correlated
formulas = {}

ldlFormula = ols_formula(cols, "ldl", ["trig"])
formulas["ldl"] = ldlFormula
imputedData.set_imputer("ldl", formula=ldlFormula)

trigFormula = ols_formula(cols, "trig", ["ldl"])
formulas["trig"] = trigFormula
imputedData.set_imputer("trig", formula=trigFormula)

# antihypetensive is a poisson...
antiHypertensiveFormula = ols_formula(cols, "antiHypertensive", ["smokingStatus_0", "smokingStatus_1","smokingStatus_2"])
antiHypertensiveFormula += '+ selfReportHtn + selfReportCurrentHtnMed + jnc8 +  meanSBP*jnc8 + meanDBP*jnc8 + meanSBP*selfReportCurrentHtnMed '
formulas["antiHypertensive"] = antiHypertensiveFormula
#imputedData.set_imputer("antiHypertensive", formula=antiHypertensiveFormula, model_class=dm.Poisson)
imputedData.set_imputer("antiHypertensive", formula=antiHypertensiveFormula)


for var in ['a1c', 'hdl', 'tot_chol', 'weight','bmi','meanSBP', 'meanDBP', 'waist', 'anyPhysicalActivity', 
            'alcoholPerWeek', 'statin', 'otherLipidLowering', 'serumCreatinine', 'urineAlbumin']:
    formula = ols_formula(cols, var, [])
    imputedData.set_imputer(var, formula=formula)
    formulas[var] = formula

imputedData.update_all(20)

### Check missing data in imputed vs. baseline dataset...generally looks pretty good

In [30]:
#CD: for checking..
### look at all imputations
#for var in ['ldl', 'trig', 'antiHypertensive', 'a1c', 'hdl', 'tot_chol', 'weight','bmi','meanSBP', 'meanDBP', 'waist',  'alcoholPerWeek', 'statin', 'otherLipidLowering', 'serumCreatinine', 'urineAlbumin']:
#    imputedData.plot_fit_obs(var)

In [31]:
# label the xs in the regressions to make the output readable...
for var in ['ldl', 'trig', 'antiHypertensive', 'a1c', 'hdl', 'tot_chol', 'weight','bmi','meanSBP', 'meanDBP', 'waist',  
            'alcoholPerWeek', 'statin', 'otherLipidLowering', 'serumCreatinine', 'urineAlbumin']:
    formula = formulas[var]
    covariates = formula.split('+')
    
    count = 1
    for cov in covariates:
        imputedData.results[var].model.exog_names[count] = cov.strip()
        count+= 1

In [32]:
#CD: checking...
#for var in ['ldl', 'trig', 'antiHypertensive', 'a1c', 'hdl', 'tot_chol', 'weight','bmi','meanSBP', 'meanDBP', 'waist',  'alcoholPerWeek', 'statin', 'otherLipidLowering', 'serumCreatinine', 'urineAlbumin']:
#   print("\n" + var + "\n")
#    print(imputedData.results[var].summary())

In [33]:
#imputedData.plot_imputed_hist("antiHypertensive", 
#                              imp_hist_args={'bins' : 7, 'alpha' : 0.5, 'color' : 'blue', 'histtype' : 'bar'},
#                             obs_hist_args={'bins' : 7, 'alpha' : 0.5, 'color' : 'green', 'histtype' : 'bar'})

In [34]:
#print(regressDF.meanSBP.describe())
#print("\n")
#print(imputedData.data.meanSBP.describe())

In [35]:
#over18.reset_index(inplace=True)
imputedAndBase = imputedData.data.join(over18, how='left', lsuffix='_x', rsuffix='_y')

In [36]:
#CD not working...
#imputedAndBase.groupby('missingSBP_x')[['age_x','age_y', 'meanSBP_x', 'meanSBP_y', 'meanDBP_x', 'meanDBP_y', 'antiHypertensive_x', 'antiHypertensive_y']].mean()

In [37]:
imputedAndBase['missingAntiHypertensive'] = imputedAndBase.antiHypertensive_y.isna()
#imputedAndBase.groupby('missingAntiHypertensive')[['age_x','age_y', 'meanSBP_x', 'meanSBP_y', 'meanDBP_x', 'meanDBP_y', 'antiHypertensive_x', 'antiHypertensive_y']].mean() #CD checking

In [38]:
### ok...so, we have problems with imputation...
### the means amongst those that are missing are too high...
### the anti hypertensives in the iputed data (_x) are lower than in the non imputed data...

### so imputation is increasing the BPS...and redsucing anti-hypertensive treatment...

In [39]:
 #imputedAndBase['missingAntiHypertensive'] = imputedAndBase.antiHypertensive_y.isna() #CD: repeated above

In [40]:
#imputedAndBase.groupby('missingAntiHypertensive')['meanSBP_x', 'meanDBP_x', 'antiHypertensive_x', 'meanSBP_y', 'meanDBP_y', 'antiHypertensive_y'].describe()

In [41]:
#imputedAndBase.groupby('missingAntiHypertensive')['antiHypertensive_x',  'antiHypertensive_y'].describe()

In [42]:
### in general...imputation is really really important for meds because we have a ton of missing...

### so, this is weird...blood pressures are much lower in those with missing meds...
### and meds are much lower in those with missing meds (boht of which seem to make sense)
### blood pressures are higher in those with missing blood pressures
### and blood pressure meds are lower in those with missing blood pressures...

### so, we end up with slightly higher incidence of hypertensino (that makes sense frmo the higher BPs in those with missing BPs — which is semi justifiable beacuse they have older ages...0
### and we have much worse BP control — that also makes sense beacuse blood pressure meds are lower in those with missing BPs, in spite of the higher BPs...

### my guess iis that the problem is that our imputation models suck when you're missing both?


In [43]:
### imputed values are slightly higher than non-imputed values...that seesm ok...mostly by bringing upu the floor

In [44]:
#%matplotlib inline
#imputedData.data.trig.hist(alpha=0.5, bins=50)
#regressDF.trig.hist(alpha=0.5, bins=50)

In [45]:
#imputedData.plot_fit_obs("antiHypertensive")

In [46]:
#len(imputedData.data)

### Export imputed dataset


In [47]:
# the imputation reset the time in US variable...we actally want the nulls there because they represent people
# that have been in the US their whole lives
imputedDataset = imputedData.data
imputedDataset.timeInUS = regressDF.timeInUS.values
imputedDataset['raceEthnicity'] = over18.raceEthnicity.values
imputedDataset['smokingStatus'] = over18.smokingStatus.values
imputedDataset['education'] = over18.education.values
imputedDataset['selfReportStrokeAge'] = over18.selfReportStrokeAge.values
imputedDataset['selfReportMIAge'] = over18.selfReportMIAge.values
imputedDataset.raceEthnicity.astype("int")
imputedDataset.drop(labels=['smokingStatus_0', 'smokingStatus_1','smokingStatus_2'],axis=1, inplace=True)
imputedDataset.drop(labels=['raceEthnicity_1', 'raceEthnicity_2','raceEthnicity_3', 
                            'raceEthnicity_4','raceEthnicity_5' ],axis=1, inplace=True)
imputedDataset.drop(labels=['education_1', 'education_2', 'education_3', 'education_4', 'education_5'],axis=1, inplace=True)
imputedDataset['diedBy2011'] = imputedDataset.diedBy2015.astype('int')

#CD added
int64_cols = int64_cols.difference(['index', 'ucod', 'completedInterview', 'missingSBP', 'htn3', 'htn4'])
imputedDataset[int64_cols] = imputedDataset[int64_cols].astype('Int64')
#imputedDataset[['age','gender','statin']] = imputedDataset[['age','gender','statin']].astype('Int64')

imputedDataset.to_csv("fullyImputedDataset.csv")

#CD added
cols_to_fix = ['raceEthnicity', 'education']
for col in cols_to_fix:
    # This turns the labels into strings so Stata doesn't complain
    imputedDataset[col] = imputedDataset[col].astype(str)
cols_to_fix = ['selfReportMI', 'selfReportStroke','year','anyPhysicalActivity','jnc8','selfReportHtn',
               'selfReportCurrentHtnMed','diedBy2015','age']
for col in cols_to_fix:
    # This turns the labels into ints in order to be consistent with how the NHANES dataframe was originally created by JB
    imputedDataset[col] = imputedDataset[col].astype(int)
    

imputedDataset.to_stata("fullyImputedDataset.dta")

#from shutil import copyfile
#copyfile("fullyImputedDataset.dta", "/Users/burke/Documents/research/bpCog/microsim/microsim/data/fullyImputedDataset.dta")

In [48]:
imputedDataset.loc[imputedDataset["age"]<1].head(20)

,a1c,age,antiHypertensive,bmi,gender,hdl,ldl,otherLipidLowering,selfReportMI,selfReportStroke,statin,timeInUS,tot_chol,trig,waist,weight,year,anyPhysicalActivity,meanSBP,meanDBP,alcoholPerWeek,serumCreatinine,urineAlbumin,jnc8,selfReportHtn,selfReportCurrentHtnMed,diedBy2015,WTINT2YR,raceEthnicity,smokingStatus,education,selfReportStrokeAge,selfReportMIAge,diedBy2011
6,4.7,0,0.0,15.70,1.0,36.0,81.0,0.0,0,0,0.0,NaN,131.0,57.0,49.0,10.8,2011,1,88.666667,64.000000,0.0,0.64,25.8,0,0,0,0,5024.464768,5,0,1,NaN,NaN,0
46,5.0,0,0.0,13.92,1.0,64.0,59.0,0.0,0,0,0.0,NaN,118.0,55.0,38.7,5.1,2011,1,85.333333,46.666667,0.0,0.95,8.4,0,0,0,0,6988.613752,4,0,1,NaN,NaN,0
55,4.8,0,0.0,13.32,1.0,55.0,71.0,0.0,0,0,0.0,NaN,172.0,48.0,44.4,7.8,2011,1,92.666667,55.333333,0.0,0.61,5.5,0,0,0,0,19442.488468,3,0,1,NaN,NaN,0
77,4.9,0,0.0,16.39,2.0,61.0,78.0,0.0,0,0,0.0,NaN,168.0,74.0,48.5,7.5,2011,1,110.000000,46.000000,0.0,0.48,4.1,0,0,0,0,20786.668002,3,0,1,NaN,NaN,0
85,5.4,0,0.0,14.56,2.0,78.0,67.0,0.0,0,0,0.0,NaN,176.0,100.0,48.3,4.9,2011,1,122.666667,51.333333,0.0,0.50,3.5,0,0,0,0,7210.384820,1,0,1,NaN,NaN,0
86,5.3,0,0.0,15.50,2.0,52.0,69.0,0.0,0,0,1.0,NaN,136.0,72.0,43.5,7.5,2011,1,98.666667,60.000000,0.0,0.72,2.1,0,0,0,0,5158.857524,4,0,1,NaN,NaN,0
101,5.0,0,0.0,15.70,2.0,46.0,87.0,0.0,0,0,0.0,NaN,145.0,63.0,46.5,10.2,2011,1,87.333333,52.666667,0.0,0.72,13.9,0,0,0,0,7367.430495,2,0,1,NaN,NaN,0
145,5.0,0,0.0,16.22,1.0,75.0,108.0,0.0,0,0,0.0,NaN,194.0,50.0,48.0,8.2,2011,1,102.000000,56.666667,0.0,0.74,23.4,0,0,0,0,4884.343512,2,0,1,NaN,NaN,0
157,5.3,0,0.0,16.50,2.0,76.0,88.0,0.0,0,0,0.0,NaN,177.0,59.0,49.6,9.7,2011,1,88.000000,53.333333,0.0,0.75,4.6,0,0,0,0,8114.787453,3,0,1,NaN,NaN,0
197,5.4,0,0.0,16.00,2.0,68.0,89.0,0.0,0,0,0.0,NaN,171.0,84.0,46.6,6.8,2011,1,100.666667,42.666667,0.0,0.68,9.4,0,0,0,0,9767.083234,1,0,1,NaN,NaN,0


In [ ]:
#CD: see how many adults and how many kids we have...
imputedDataset.groupby(imputedDataset["age"]<18).count()

In [49]:
imputedDataset.describe()

,a1c,age,antiHypertensive,bmi,gender,hdl,ldl,otherLipidLowering,selfReportMI,selfReportStroke,statin,timeInUS,tot_chol,trig,waist,weight,year,anyPhysicalActivity,meanSBP,meanDBP,alcoholPerWeek,serumCreatinine,urineAlbumin,jnc8,selfReportHtn,selfReportCurrentHtnMed,diedBy2015,WTINT2YR,selfReportStrokeAge,selfReportMIAge,diedBy2011
count,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,13405.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.000000,70190.0,70190.000000,1.318000e+03,1441.000000,70190.0
mean,5.506314,31.775224,0.364653,25.010487,1.505499,54.528651,103.807124,0.025645,0.024391,0.022667,0.133438,8.274077,178.677347,109.068799,84.568775,62.494519,2010.886024,0.839607,115.804630,65.049584,8.850178,0.809080,42.411250,0.098817,0.205442,0.155250,0.0,30571.525412,2.181121e+03,2206.195003,0.0
std,0.922460,24.882077,0.804287,8.004197,0.499973,15.703199,35.094985,0.164868,0.154261,0.148841,0.341054,16.878595,40.973404,92.357247,24.179140,32.237469,3.975893,0.366973,18.798546,12.512569,15.510975,0.378174,296.857836,0.298419,0.404027,0.362145,0.0,30660.423044,1.441675e+04,14505.381377,0.0
min,2.000000,0.000000,0.000000,11.500000,1.000000,6.000000,9.000000,0.000000,0.000000,0.000000,0.000000,1.000000,59.000000,10.000000,37.800000,2.600000,2005.000000,0.000000,64.666667,40.000000,0.000000,0.140000,0.210000,0.000000,0.000000,0.000000,0.0,1224.968032,5.397605e-79,1.000000,0.0
25%,5.100000,10.000000,0.000000,18.310000,1.000000,43.000000,79.000000,0.000000,0.000000,0.000000,0.000000,3.000000,149.000000,60.000000,65.800000,38.000000,2007.000000,1.000000,102.666667,56.000000,0.000000,0.620000,4.500000,0.000000,0.000000,0.000000,0.0,11505.905217,4.800000e+01,47.000000,0.0
50%,5.300000,26.000000,0.000000,24.210000,2.000000,52.000000,99.000000,0.000000,0.000000,0.000000,0.000000,5.000000,173.000000,87.000000,86.000000,65.500000,2011.000000,1.000000,112.666667,64.666667,0.000000,0.760000,8.600000,0.000000,0.000000,0.000000,0.0,19822.874818,6.050000e+01,57.000000,0.0
75%,5.600000,53.000000,0.000000,29.770000,2.000000,63.000000,124.000000,0.000000,0.000000,0.000000,0.000000,7.000000,203.000000,130.000000,101.800000,83.500000,2015.000000,1.000000,125.333333,73.333333,14.000000,0.920000,17.800000,0.000000,0.000000,0.000000,0.0,35971.005811,7.100000e+01,68.000000,0.0
max,17.800000,85.000000,7.000000,130.210000,2.000000,226.000000,375.000000,2.000000,1.000000,1.000000,2.000000,99.000000,813.000000,4233.000000,179.000000,371.000000,2017.000000,1.000000,270.000000,138.666667,252.000000,17.800000,24440.000000,1.000000,1.000000,1.000000,0.0,433085.005262,9.999900e+04,99999.000000,0.0


In [60]:
imputedDataset.loc[imputedDataset["age"]<18].min()

a1c                                3.8
age                                  0
antiHypertensive                   0.0
bmi                               11.5
gender                             1.0
hdl                               11.0
ldl                                9.0
otherLipidLowering                 0.0
selfReportMI                         0
selfReportStroke                     0
statin                             0.0
timeInUS                           1.0
tot_chol                          66.0
trig                              10.0
waist                             37.8
weight                             2.6
year                              2005
anyPhysicalActivity                  1
meanSBP                           74.0
meanDBP                           40.0
alcoholPerWeek                     0.0
serumCreatinine                   0.14
urineAlbumin                      0.21
jnc8                                 0
selfReportHtn                        0
selfReportCurrentHtnMed  

In [61]:
imputedDataset.loc[imputedDataset["age"]<18].max()

a1c                                 15.6
age                                   17
antiHypertensive                     4.0
bmi                                62.08
gender                               2.0
hdl                                179.0
ldl                                257.0
otherLipidLowering                   2.0
selfReportMI                           0
selfReportStroke                       0
statin                               1.0
timeInUS                            99.0
tot_chol                           440.0
trig                              1562.0
waist                              167.2
weight                             215.3
year                                2017
anyPhysicalActivity                    1
meanSBP                       173.333333
meanDBP                            104.0
alcoholPerWeek                       0.0
serumCreatinine                    12.48
urineAlbumin                      7080.0
jnc8                                   1
selfReportHtn   

In [51]:
imputedDataset.loc[imputedDataset["age"]>=18].describe()

,a1c,age,antiHypertensive,bmi,gender,hdl,ldl,otherLipidLowering,selfReportMI,selfReportStroke,statin,timeInUS,tot_chol,trig,waist,weight,year,anyPhysicalActivity,meanSBP,meanDBP,alcoholPerWeek,serumCreatinine,urineAlbumin,jnc8,selfReportHtn,selfReportCurrentHtnMed,diedBy2015,WTINT2YR,selfReportStrokeAge,selfReportMIAge,diedBy2011
count,42143.00000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,11781.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.000000,42143.0,42143.000000,1.318000e+03,1441.000000,42143.0
mean,5.72925,47.939539,0.577984,28.948596,1.514558,53.244881,114.177823,0.039390,0.040624,0.037752,0.210379,8.660895,191.368389,126.749258,98.677365,80.959096,2010.999668,0.732862,123.767063,70.372090,14.740147,0.900803,49.818479,0.163823,0.340341,0.258358,0.0,38541.768622,2.181121e+03,2206.195003,0.0
std,1.07342,18.924719,0.964382,7.056966,0.499794,16.210146,36.650813,0.203233,0.197419,0.190599,0.408977,17.051132,42.279498,108.045902,16.904625,21.950297,3.948929,0.442470,18.984300,11.613836,17.716962,0.438994,360.538742,0.370119,0.473830,0.437737,0.0,35112.789868,1.441675e+04,14505.381377,0.0
min,2.00000,18.000000,0.000000,13.180000,1.000000,6.000000,9.000000,0.000000,0.000000,0.000000,0.000000,1.000000,59.000000,10.000000,42.400000,5.300000,2005.000000,0.000000,64.666667,40.000000,0.000000,0.160000,0.210000,0.000000,0.000000,0.000000,0.0,1339.046608,5.397605e-79,1.000000,0.0
25%,5.20000,31.000000,0.000000,24.010000,1.000000,42.000000,88.000000,0.000000,0.000000,0.000000,0.000000,3.000000,162.000000,69.000000,86.600000,65.700000,2007.000000,0.000000,110.666667,62.666667,7.000000,0.710000,4.300000,0.000000,0.000000,0.000000,0.0,16390.650364,4.800000e+01,47.000000,0.0
50%,5.50000,47.000000,0.000000,27.830000,2.000000,51.000000,111.000000,0.000000,0.000000,0.000000,0.000000,6.000000,188.000000,102.000000,97.400000,77.900000,2011.000000,1.000000,120.666667,70.000000,7.000000,0.850000,8.400000,0.000000,0.000000,0.000000,0.0,25345.444945,6.050000e+01,57.000000,0.0
75%,5.90000,63.000000,1.000000,32.450000,2.000000,62.000000,136.000000,0.000000,0.000000,0.000000,0.000000,7.000000,217.000000,152.000000,108.600000,92.400000,2015.000000,1.000000,133.333333,77.333333,21.000000,1.000000,17.600000,0.000000,1.000000,1.000000,0.0,48609.808693,7.100000e+01,68.000000,0.0
max,17.80000,85.000000,7.000000,130.210000,2.000000,226.000000,375.000000,2.000000,1.000000,1.000000,2.000000,99.000000,813.000000,4233.000000,179.000000,371.000000,2017.000000,1.000000,270.000000,138.666667,252.000000,17.800000,24440.000000,1.000000,1.000000,1.000000,0.0,433085.005262,9.999900e+04,99999.000000,0.0


In [ ]:
#check a few things....

In [ ]:
imputedDataset.loc[imputedDataset["age"]<20,]["smokingStatus"].value_counts()

In [ ]:
imputedDataset.loc[imputedDataset["age"]<20,]["education"].value_counts()

In [ ]:
imputedDataset.loc[imputedDataset["age"]<20,]["alcoholPerWeek"].value_counts()

In [ ]:
imputedDataset.loc[imputedDataset["age"]<20,]["anyPhysicalActivity"].value_counts()

In [ ]:
#imputedDataset.meanSBP.hist()

In [ ]:
imputedDatasetLoaded = pd.read_stata('fullyImputedDataset.dta')

In [ ]:
imputedDatasetLoaded.columns

In [ ]:
imputedDatasetLoaded.head()

In [ ]:
#the two dataframes should have identical types
microsimDir = "/Users/deligkaris.1/OneDrive - The Ohio State University Wexner Medical Center/MICROSIM/CODE/microsim"
os.chdir(microsimDir)
nhanesDf = pd.read_stata("microsim/data/fullyImputedDataset.dta")
nhanesDf.head()

In [ ]:
nhanesDf.shape

In [ ]:
print(nhanesDf.loc[nhanesDf["age"]<1].head(50))